Входные данные, в будущем лучше грузить из файла

In [1]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

In [2]:
# My library imports
from fluid import Fluid
from function_dp import vniigaz_vertical_one, vniigaz_vertical_two, vniigaz_inclined, vniigaz_horizontal
from function_dp import lyamda

In [3]:
# Исходные значения для расчёта
# lyamda=0.0140
diameter=0.1000
alfa=45
v=1.95E-03
u=5.643
Ro_water=1000.00
Ro_gas=19.24764133
pressure=2.896463361
sigma=0.072
q_liquid=1.52778E-05
q_gas=1.273148148
delta_0 = 0.0078 # шероховатость трубы в мм

# # -- Тестовые расчёты функции ВНИИГАЗ
# a parametrs for horizontal:
# idx = 465
# alfa = well_propertis.loc[idx, 'alfa']
# # q_liquid = 0.000138889
# q_liquid = ((Q_gas_std*1000/86400)*wgr)
# diameter = 0.1
# v = well_propertis.loc[idx, 'v']
# u = well_propertis.loc[idx, 'u']
# Ro_gas = well_propertis.loc[idx, 'Ro_gas']
# q_gas = well_propertis.loc[idx, 'Qgas_pt']
# print(alfa)

func_lyambda = lyamda(delta0=0.0078, diameter=0.1, 
                      velocity_liquid=0.00194523, velocity_gas=5.643, 
                      density_liquid=1000, density_gas=19.24764133, 
                      sigma=0.072, viscosity_gas=0.000012)
print("lyamda:", func_lyambda)

horizont = vniigaz_horizontal(0.014, diameter, alfa, v, u, Ro_water, Ro_gas, pressure, sigma, q_liquid)
print("dp_dz from vniigaz_horizont:", horizont)

# inclined = vniigaz_inclined(lyamda, diameter, alfa, v, u, Ro_water, Ro_gas, pressure, sigma, q_liquid)
# print("dp_dz from vniigaz_inclined:", inclined)

# vertical = vniigaz_vertical_one(lyamda, diameter, v, u, Ro_water, Ro_gas, pressure, sigma, q_liquid)
# print("dp_dz from vniigaz_vertical_one:", vertical)

# vertical_two = vniigaz_vertical_two(lyamda, diameter, v, u, Ro_water, Ro_gas, pressure, sigma, q_liquid)
# print("dp_dz from vniigaz_two:", vertical_two)


lyamda: 0.013243343183010918, delta: 0.007577802117391073, delta1: 0.0032879173357725304, eps: 7.577802117391073e-05
lyamda: 0.013243343183010918
dp_dz from vniigaz_horizont: 644.4243111467007


In [9]:
# Standard conditions
Pstd = 101325 #Pa
Tstd = 273.15 + 20 #K

# Fluid
Ro_gas_std = 0.67 # кг/м3
Ro_water_std = 1000 # кг/м3
sigma = 0.072 # Н/м, поверхностное натяжение

# Reservoir
permability = 10*10**-12 # м2
pay_thickness = 10 # м

reservoir_temp = 305.15 # K
wellhead_temp = 293.15 # K
reservoir_pressure = 4.65 # МПа

# Data for gas-liquid flow
Q_gas_std = 100 # тыс.м3/сутки
diameter = 0.1 # м, диаметр трубы
delta0 = 0.0078 # мм, шероховатость трубы
# wgr = 12*10**-5 # м3/м3, водогазовый фактор
wgr = 1e-5 # м3/м3, водогазовый фактор

In [4]:
# Well properties
# Download inclinometry data
file_path_incl = 'include/incl_17.dev'  # Update this path to match your actual file location
with open(file_path_incl, 'r', encoding='utf-8') as inclinometry_file:
    well_data = pd.read_csv(inclinometry_file, comment='#', sep=r'\s+')

# Download inclinometry data_horizontal
file_path_incl_horizontal = 'include/incl_10102.dev'  # Update this path to match your actual file location
with open(file_path_incl_horizontal, 'r', encoding='utf-8') as inclinometry_file_horizontal:
    well_data_hor = pd.read_csv(inclinometry_file_horizontal, comment='#', sep=r'\s+')

# Download inclinometry data full vertical well
file_path_incl_horizontal = 'include/incl_vertical.dev'  # Update this path to match your actual file location
with open(file_path_incl_horizontal, 'r', encoding='utf-8') as inclinometry_file_vert:
    well_data_vert = pd.read_csv(inclinometry_file_vert, comment='#', sep=r'\s+')

# Download GDI
file_path_incl_GDI = 'include/GDI.inc'  # Update this path to match your actual file location
with open(file_path_incl_GDI, 'r', encoding='utf-8') as GDI_data:
    GDI_data = pd.read_csv(GDI_data, comment='#', sep=r'\s+')

# Gas properties
# Download PVT data
file_path_pvt = 'include/PVT.inc'  # Update this path to match your actual file location
with open(file_path_pvt, 'r', encoding='utf-8') as pvt_file:
    pvt_data = pd.read_csv(pvt_file, comment='#', sep=r'\s+')

# Test fuid properties
Ro_gas_std = 0.6799
xa = 0.8858
xy = 0.0668
fluid = Fluid(Ro_gas_std, xa, xy, pvt_data)

In [5]:
# Расчёт градиента давления (dp/dx) от забоя до устья

well_propertis = pd.DataFrame({
    'MD': well_data_vert['MD'],
    'TVD': well_data_vert['TVD'],
    'alfa': well_data_vert['INCL'],
    'Diameter': diameter,
    'T': np.nan,
    'z': np.nan,
    'Ro_gas': np.nan,
    'Qgas_pt': np.nan,
    'Dp_dz': np.nan,
})

for idx in well_propertis.index[::-1]:

    # Заполнение параметра Т в зависимости от глубины скважины
    if idx == 0:
        well_propertis.loc[idx, 'T'] = wellhead_temp
    elif idx == len(well_propertis)-1:
        well_propertis.loc[idx, 'T'] = reservoir_temp
    else:
        well_propertis.loc[idx, 'T'] = reservoir_temp - (reservoir_temp - wellhead_temp) * (well_propertis.loc[idx, 'TVD'] / well_propertis['TVD'].iloc[-1])

    # Заполнение параметра Pi в зависимости от глубины скважины
    if idx == len(well_propertis)-1:
        well_propertis.loc[idx, 'Pi'] = reservoir_pressure

    tempreture = well_propertis.loc[idx, 'T'] # К
    pressure = well_propertis.loc[idx, 'Pi'] # МПа

    # Расчёт z
    if not pd.isna(well_propertis.loc[idx, 'Pi']):
        well_propertis.loc[idx, 'z'] = fluid.get_Z(pressure, tempreture)
        well_propertis.loc[idx, 'Ro_gas'] = fluid.get_ro(well_propertis.loc[idx, 'Pi'], well_propertis.loc[idx, 'T'])
        well_propertis.loc[idx, 'u'] = (Q_gas_std*1000/86400) * Ro_gas_std / well_propertis.loc[idx, 'Ro_gas'] / (np.pi*well_propertis.loc[idx, 'Diameter']**2/4)
        well_propertis.loc[idx, 'v'] = ((Q_gas_std*1000/86400)*wgr)/(np.pi*well_propertis.loc[idx, 'Diameter']**2/4)
        well_propertis.loc[idx, 'Qgas_pt'] = (Q_gas_std*1000/86400) * fluid.get_Bg(pressure, tempreture)
        func_lyambda = lyamda(delta0=delta0, diameter=well_propertis.loc[idx, 'Diameter'], 
                      velocity_liquid=well_propertis.loc[idx, 'v'], velocity_gas=well_propertis.loc[idx, 'u'], 
                      density_liquid=1000, density_gas=well_propertis.loc[idx, 'Ro_gas'], 
                      sigma=0.072, viscosity_gas= fluid.get_viscosity(well_propertis.loc[idx, 'Pi']))
        if well_propertis.loc[idx, 'alfa'] < 84:
            well_propertis.loc[idx, 'Dp_dz'] = vniigaz_inclined(func_lyambda, diameter, well_propertis.loc[idx, 'alfa'], well_propertis.loc[idx, 'v'], well_propertis.loc[idx, 'u'], Ro_water_std, well_propertis.loc[idx, 'Ro_gas'], pressure, sigma, ((Q_gas_std*1000/86400)*wgr))
        else:
            well_propertis.loc[idx, 'Dp_dz'] = vniigaz_horizontal(func_lyambda, diameter, well_propertis.loc[idx, 'alfa'], well_propertis.loc[idx, 'v'], well_propertis.loc[idx, 'u'], Ro_water_std, well_propertis.loc[idx, 'Ro_gas'], pressure, sigma, ((Q_gas_std*1000/86400)*wgr))
        if idx > 0:
            well_propertis.loc[idx, 'Pi1'] = well_propertis.loc[idx, 'Pi'] - (well_propertis.loc[idx, 'Dp_dz'] * (well_propertis.loc[idx, 'MD'] - well_propertis.loc[idx-1, 'MD']))*10**-6     
            well_propertis.loc[idx-1, 'Pi'] = well_propertis.loc[idx, 'Pi1']

NameError: name 'reservoir_temp' is not defined

In [6]:
# Функция для расчёта забойного давления BHP по устьевому THP и расходу газа 

def well_BHP(well_data: pd, fluid: Fluid, THP, Temp_THP, Temp_BHP, Qgas, WGR, diameter, delta0, sigma, density_liquid):
    """
    Параметры:
        well_data - DataFrame;
        fluid - class;
        THP [K];
        Qgas [ст. тыс.м3/сут];
        WGR [м3/м3];
        diamert [м];
        delta0 [мм];
        sigma [Н/м]
        density_liquid [кг/м3]
    Возвращает:
        BHP -> МПа
    """

    well_df = pd.DataFrame({
        'MD': well_data['MD'],
        'TVD': well_data['TVD'],
        'alfa': well_data['INCL'],
        'D': diameter, # TODO: сделать возможность задания конструкции
        'Dp_dz': np.nan,
        'P': np.nan
    })

    for idx in well_df.index:
        if idx == 0:
            well_df.loc[idx, 'P'] = THP
        if idx == len(well_df)-1:
            break

        # FIXME: Вставить функцию расчёта лямбды
        temp_idx = Temp_THP + (Temp_BHP-Temp_THP)/well_df['TVD'].iloc[-1]*well_df.loc[idx, 'TVD'] # К
        density_gas_idx = fluid.get_ro(well_df.loc[idx, 'P'], temp_idx) # кг/м3
        velocity_liquid_idx = ((Qgas*1000/86400)*WGR) / (np.pi*well_df.loc[idx, 'D']**2/4) # м/с
        velocity_gas_idx = (Qgas*1000/86400) * fluid.get_fvf(well_df.loc[idx, 'P']) / (np.pi * well_df.loc[idx, 'D']**2 / 4) # м/с

        lyamda_idx = lyamda(delta0=delta0, diameter=well_df.loc[idx, 'D'], 
              velocity_liquid=velocity_liquid_idx, velocity_gas=velocity_gas_idx, 
              density_liquid=density_liquid, density_gas=density_gas_idx, 
              sigma=sigma, viscosity_gas=fluid.get_viscosity(well_df.loc[idx, 'P']))

        if well_df.loc[idx, 'alfa'] > 84:
            well_df.loc[idx, 'Dp_dz'] = vniigaz_horizontal(lyamda_idx, well_df.loc[idx, 'D'], well_df.loc[idx, 'alfa'],
                                                            velocity_liquid_idx, velocity_gas_idx, density_liquid,
                                                              density_gas_idx, well_df.loc[idx, 'P'],
                                                                sigma, ((Qgas*1000/86400)*WGR)) # Па/м
        else:
            well_df.loc[idx, 'Dp_dz'] = vniigaz_inclined(lyamda_idx, well_df.loc[idx, 'D'], well_df.loc[idx, 'alfa'],
                                                          velocity_liquid_idx, velocity_gas_idx, density_liquid,
                                                            density_gas_idx, well_df.loc[idx, 'P'],
                                                              sigma, ((Qgas*1000/86400)*WGR)) # Па/м

        well_df.loc[idx+1,'P'] = well_df.loc[idx,'P'] + well_df.loc[idx, 'Dp_dz'] * 10**-6 * (well_df.loc[idx+1, 'MD'] - well_df.loc[idx, 'MD']) # МПа

    return well_df['P'].iloc[-1] # МПа

In [7]:
# TODO: Функция адаптации def well_BHP(lyamda, density_liquid) от лямдбы и плотности ГЖС на фактические резльтаты исследований GDI

def adapt_gdi(GDI_data, delta0, density_liquid):
        df_data = pd.DataFrame({
        'THP': GDI_data['THP'], # перевод в МПа
        'FLO': GDI_data['FLO']/1000, # перевод в ст. тыс.м3/сут
        'WGR': GDI_data['WGR'], # м3/м3
        
        'BHP': GDI_data['BHP'], # перевод в МПа
        'dp_gdi': GDI_data['dp'], # перевод в МПа
        
        'BHP_calc': np.nan,
        'dp_calc': np.nan,
        'd_bhp': np.nan
        })

        for idx in df_data.index:
                df_data.loc[idx,'BHP_calc'] = well_BHP(well_data_vert, fluid, 
                                                       df_data.loc[idx, 'THP']/10, wellhead_temp, reservoir_temp, 
                                                       df_data.loc[idx, 'FLO'], df_data.loc[idx, 'WGR'], diameter, 
                                                       delta0, 0.072, density_liquid)*10
                
                df_data.loc[idx,'dp_calc'] = df_data.loc[idx, 'BHP_calc'] - df_data.loc[idx, 'THP']

                df_data.loc[idx,'d_bhp'] = df_data.loc[idx,'BHP_calc'] - df_data.loc[idx,'BHP']
        
        r2 = r2_score(df_data['BHP'], df_data['BHP_calc'])
    
        return df_data
    # return lyamda, density_liquid

In [ ]:
# delta0 = 5.276637718751923e-14
delta0 = 0.00001
density_liquid = 849.9190696690689
adapt_gdi(GDI_data, delta0, density_liquid)

lyamda: 0.059882672428302294, delta: 0.00039531644076848453, delta1: 0.0026640051070238235, eps: 3.9531644076848455e-06
lyamda: 0.059886060053626414, delta: 0.0003981924809555253, delta1: 0.0026739051225517178, eps: 3.9819248095552534e-06
lyamda: 0.0598894750339511, delta: 0.00040100845132267206, delta1: 0.0026835628574809352, eps: 4.01008451322672e-06
lyamda: 0.05989289684028437, delta: 0.00040386097762539337, delta1: 0.0026933105380585014, eps: 4.038609776253934e-06
lyamda: 0.059896325539633, delta: 0.0004067507556700249, delta1: 0.002703149552285498, eps: 4.067507556700249e-06
lyamda: 0.05989976120008261, delta: 0.000409678498828259, delta1: 0.0027130813182536903, eps: 4.096784988282589e-06
lyamda: 0.05990320389082222, delta: 0.00041264493859163376, delta1: 0.0027231072849885573, eps: 4.126449385916337e-06
lyamda: 0.05990665800535871, delta: 0.0004156299594019452, delta1: 0.0027331588022405083, eps: 4.156299594019452e-06
lyamda: 0.059910141926124746, delta: 0.00041854392979826926, d

,THP,FLO,WGR,BHP,dp_gdi,BHP_calc,dp_calc,d_bhp
0,31,127.0,0.000032,39.515795,8.52,35.757064,4.757064,-3.758732
1,2,509.0,0.000055,29.333288,27.33,47.667504,45.667504,18.334216
2,17,193.0,0.000081,24.207471,7.21,27.362676,10.362676,3.155205
3,54,241.0,0.000641,79.431721,25.43,77.708168,23.708168,-1.723553
4,118,365.0,0.000003,129.453806,11.45,131.845002,13.845002,2.391196
5,54,586.0,0.000097,72.089533,18.09,81.768610,27.768610,9.679077
6,200,260.0,0.000020,231.002102,31.00,220.032231,20.032231,-10.969871


In [11]:
# функция для подстановки параметров
def objective(params):
    lyamda, density_liquid = params

    # Ограничения (важно!)
    if lyamda <= 0 or density_liquid <= 0:
        return 1e6

    df = adapt_gdi(GDI_data, lyamda, density_liquid)

    mse = mean_squared_error(df['BHP'], df['BHP_calc'])
    return mse  # минимизируем ошибку

In [12]:
# Вызов функции адаптации
initial_guess = [0.02, 900]

result = minimize(
    objective,
    initial_guess,
    method='Nelder-Mead'
)

opt_lyamda, opt_density = result.x

print(f'Оптимальные параметры:')
print(f'lyamda = {opt_lyamda}')
print(f'density_liquid = {opt_density}')

Оптимальные параметры:
lyamda = 5.276637718751923e-14
density_liquid = 849.9190696690689


In [13]:
q_gas_list = [50, 100, 150, 250, 400, 600]
wgr_list = [0, 1e-06, 1e-05, 1e-04]
thp_list = [1, 2, 5, 10, 15, 20]


In [14]:
# FIXME: Привести в порядок, дописать код для полноценого создания VFP

def generate_vfp_table(q_gas_list, wgr_list, thp_list,
                       well_data, fluid,
                       Temp_THP, Temp_BHP,
                       diameter, lyamda, sigma, density_liquid):

    rows = []

    # ВАЖНО: порядок как в VFP (WGR -> THP)
    for i_wgr, wgr in enumerate(wgr_list, start=1):
        for i_thp, thp in enumerate(thp_list, start=1):

            row = {
                'THP_i': i_thp,
                'WGR_i': i_wgr,
                'GFR_i': 1,
                'ALQ_i': 1,
            }

            # считаем BHP для каждого расхода газа
            for i_q, q in enumerate(q_gas_list, start=1):
                bhp = well_BHP(
                    well_data,
                    fluid,
                    thp,
                    Temp_THP,
                    Temp_BHP,
                    q,
                    wgr,
                    diameter,
                    lyamda,
                    sigma,
                    density_liquid
                )

                # МПа → бар
                row[f'BHP_q{i_q}'] = bhp * 10

            rows.append(row)

    df = pd.DataFrame(rows)
    return df

In [15]:
vfp_table = generate_vfp_table(q_gas_list, wgr_list, thp_list, well_data_vert, fluid, 293, 310, 0.1, 0.0114, 0.072, 1000)

In [16]:
vfp_table.to_csv('vfp_table')

In [17]:
# FIXME: полностью переписать архитектуру вызовов!

# ============================================================
# КОНСТАНТЫ И ПАРАМЕТРЫ
# ============================================================

# Standard conditions
Pstd = 101325  # Pa
Tstd = 273.15 + 20  # K

# Fluid properties
Ro_gas_std = 0.67  # кг/м3
Ro_water_std = 1000  # кг/м3
sigma = 0.072  # Н/м

# Well parameters
diameter = 0.1  # м
lyamda = 0.02

# Temperature profile
wellhead_temp = 293.15  # K
reservoir_temp = 305.15  # K

# Load inclinometry data
file_path_incl_horizontal = 'd:/Python_2025/Gas_liquid_flow/исходные_данные/vertical_well.dev'
with open(file_path_incl_horizontal, 'r', encoding='utf-8') as inclinometry_file_vertical:
    well_data_hor = pd.read_csv(inclinometry_file_vertical, comment='#', sep=r'\s+')

# Maximum TVD (глубина по вертикали) - bottom hole depth
max_TVD = well_data_hor['TVD'].iloc[-1]

# Load PVT data
file_path_pvt = 'd:/Python_2025/Gas_liquid_flow/исходные_данные/PVT.inc'
with open(file_path_pvt, 'r', encoding='utf-8') as pvt_file:
    pvt_data = pd.read_csv(pvt_file, comment='#', sep=r'\s+')

# Create fluid object
xa = 0.8858
xy = 0.0668
fluid_obj = fluid.Fluid(Ro_gas_std, xa, xy, pvt_data)

# ============================================================
# ВХОДНЫЕ ДАННЫЕ ДЛЯ VFP ТАБЛИЦЫ
# ============================================================

# Номер таблицы
table_number = 1

# Дебиты газа FLO (м3/сут) - в возрастающем порядке
gas_rates = np.array([30000, 50000, 100000, 200000])

# Устьевые давления THP (бар) - в возрастающем порядке
thp_bar = np.array([5, 30, 80])
thp_pa = thp_bar * 1e5

# Водогазовый фактор WFR (WGR) (м3/м3) - в возрастающем порядке
wgr_values = np.array([0.000001, 0.00001, 0.0001])

# GFR (OGR) - не используем, но нужно для формата
ogr_values = np.array([0.0])

# ALQ (GRAT) - не используем
grat_values = np.array([0.0])

# ============================================================
# ФУНКЦИЯ ДЛЯ РАСЧЁТА BHP ПО ИЗВЕСТНОМУ THP
# ============================================================

def calculate_bhp_from_thp(thp_pa, Q_gas_std_m3_per_day, wgr, well_data, 
                           diameter, lyamda, Ro_water_std, sigma, 
                           fluid_obj, wellhead_temp, reservoir_temp,
                           Pstd, Tstd, Ro_gas_std):
    """
    Расчёт забойного давления BHP по известному устьевому давлению THP.
    Возвращает BHP в барах.
    """
    
    # Конвертируем дебит газа в м3/с
    Q_gas_std = Q_gas_std_m3_per_day / 86400
    
    # Расход жидкости (воды) при стандартных условиях
    Q_water_std = Q_gas_std * wgr
    
    # Создаём копию данных для расчёта
    df = well_data.copy()
    
    # Заполняем температуру
    for idx in df.index:
        if idx == 0:
            df.loc[idx, 'T'] = wellhead_temp
        elif idx == len(df)-1:
            df.loc[idx, 'T'] = reservoir_temp
        else:
            df.loc[idx, 'T'] = reservoir_temp - (reservoir_temp - wellhead_temp) * (df.loc[idx, 'TVD'] / df['TVD'].iloc[-1])
    
    # Начальное давление на устье (первая точка)
    df.loc[0, 'P'] = thp_pa
    
    # Идём от устья к забою
    for idx in range(len(df)-1):
        pressure = df.loc[idx, 'P']
        temperature = df.loc[idx, 'T']
        alfa = df.loc[idx, 'INCL']
        
        # PVT свойства
        try:
            pressure_mpa = pressure / 1e6
            Bg = fluid_obj.get_Bg(pressure_mpa, temperature)
            if np.isnan(Bg) or np.isinf(Bg):
                Bg = 0.01
        except:
            Bg = 0.01
        
        # Плотность газа через Bg
        Ro_gas = Ro_gas_std / Bg
        
        # Расход газа при скважинных условиях
        Q_gas_pt = Q_gas_std * Bg
        
        # Площадь сечения
        area = np.pi * diameter**2 / 4
        
        # Скорости
        u_gas = Q_gas_pt / area if Q_gas_pt > 0 else 0
        
        if Q_water_std > 0:
            Q_water_pt = Q_water_std
            u_liq = Q_water_pt / area
        else:
            Q_water_pt = 0.0
            u_liq = 0.0
        
        # Расчёт градиента
        try:
            if alfa < 85:
                dp_dz = vniigaz_inclined(
                    lyamda, diameter, alfa,
                    u_liq, u_gas,
                    Ro_water_std, Ro_gas,
                    pressure, sigma,
                    Q_water_pt, Q_gas_pt
                )
            else:
                dp_dz = vniigaz_horizontal(
                    lyamda, diameter, alfa,
                    u_liq, u_gas,
                    Ro_water_std, Ro_gas,
                    pressure, sigma,
                    Q_water_pt, Q_gas_pt
                )
            
            if np.isnan(dp_dz) or np.isinf(dp_dz):
                dp_dz = 1000
        except:
            dp_dz = 1000
        
        # Переход к следующей точке
        delta_MD = df.loc[idx+1, 'MD'] - df.loc[idx, 'MD']
        pressure_next = pressure + dp_dz * delta_MD
        
        if np.isnan(pressure_next) or np.isinf(pressure_next):
            return 1e10
        
        df.loc[idx+1, 'P'] = pressure_next
        
        if pressure_next > 1e10:
            return 1e10
    
    # Возвращаем BHP в барах
    bhp_pa = df.loc[len(df)-1, 'P']
    bhp_bar = bhp_pa / 1e5
    
    return bhp_bar


# ============================================================
# РАСЧЁТ ВСЕХ КОМБИНАЦИЙ
# ============================================================

print("=" * 70)
print("ГЕНЕРАЦИЯ VFP ТАБЛИЦЫ")
print("=" * 70)
print(f"Таблица №{table_number}")
print(f"Глубина забоя: {max_TVD:.0f} м")
print(f"Дебиты газа (FLO): {gas_rates}")
print(f"Устьевые давления (THP): {thp_bar} бар")
print(f"Водогазовый фактор (WFR): {wgr_values}")
print("=" * 70)

# Размерности
NFLO = len(gas_rates)      # количество дебитов
NTHP = len(thp_bar)        # количество THP
NWFR = len(wgr_values)     # количество WFR
NGFR = len(ogr_values)     # количество GFR (у нас 1)
NALQ = len(grat_values)    # количество ALQ (у нас 1)

print(f"Всего комбинаций: {NTHP} × {NWFR} × {NGFR} × {NALQ} = {NTHP * NWFR * NGFR * NALQ}")
print(f"Каждая запись содержит {NFLO} значений BHP")
print(f"Всего расчётов: {NFLO * NTHP * NWFR * NGFR * NALQ}")
print("=" * 70)

# Массив для хранения результатов: [NTHP][NWFR][NGFR][NALQ][NFLO]
# Инициализируем массив BHP значениями 1e10 (признак ошибки)
bhp_results = np.full((NTHP, NWFR, NGFR, NALQ, NFLO), 1e10)

total_cases = NFLO * NTHP * NWFR * NGFR * NALQ
case_counter = 0

# Вложенные циклы по всем комбинациям
for i_thp in range(NTHP):
    thp = thp_bar[i_thp]
    for i_wfr in range(NWFR):
        wgr = wgr_values[i_wfr]
        for i_gfr in range(NGFR):
            ogr = ogr_values[i_gfr]  # не используется
            for i_alq in range(NALQ):
                grat = grat_values[i_alq]  # не используется
                
                # Для каждого дебита газа
                for i_flo in range(NFLO):
                    Q_gas = gas_rates[i_flo]
                    case_counter += 1
                    
                    print(f"Расчёт {case_counter}/{total_cases}: THP={thp} бар, Qgas={Q_gas} м3/сут, WGR={wgr:.6f}")
                    
                    try:
                        bhp = calculate_bhp_from_thp(
                            thp * 1e5, Q_gas, wgr, well_data_hor,
                            diameter, lyamda, Ro_water_std, sigma,
                            fluid_obj, wellhead_temp, reservoir_temp,
                            Pstd, Tstd, Ro_gas_std
                        )
                        bhp_results[i_thp, i_wfr, i_gfr, i_alq, i_flo] = bhp
                        print(f"  BHP = {bhp:.4f} бар")
                    except Exception as e:
                        print(f"  ОШИБКА: {e}")
                        bhp_results[i_thp, i_wfr, i_gfr, i_alq, i_flo] = 1e10

import pandas as pd

# После завершения всех расчётов, создаём DataFrame с подписями

# Получаем форму массива
NTHP, NWFR, NGFR, NALQ, NFLO = bhp_results.shape

# Создаём список для сбора данных
rows_data = []

# Заголовок для пояснения
print("Формирование VFP-подобной таблицы...")

for i_thp in range(NTHP):
    for i_wfr in range(NWFR):
        for i_gfr in range(NGFR):
            for i_alq in range(NALQ):
                # Получаем значения BHP для всех дебитов
                bhp_values = bhp_results[i_thp, i_wfr, i_gfr, i_alq, :]
                
                # Создаём строку с описанием параметров
                row = {
                    'THP_N': i_thp + 1,
                    'WFR_N': i_wfr + 1,
                    'GFR_N': i_gfr + 1,
                    'ALQ_N': i_alq + 1,
                    'THP_бар': thp_bar[i_thp],
                    'WGR_м3/м3': wgr_values[i_wfr],
                    'GFR_OGR': ogr_values[i_gfr],
                    'ALQ_GRAT': grat_values[i_alq]
                }
                
                # Добавляем BHP для каждого дебита газа
                for i_flo in range(NFLO):
                    row[f'BHP_FLO_{i_flo+1}_Qgas={gas_rates[i_flo]:.0f}'] = bhp_values[i_flo]
                
                rows_data.append(row)

import pandas as pd

# После завершения всех расчётов создаём DataFrame
rows_data = []

for i_thp in range(NTHP):
    for i_wfr in range(NWFR):
        for i_gfr in range(NGFR):
            for i_alq in range(NALQ):
                # Получаем BHP для всех дебитов
                bhp_values = bhp_results[i_thp, i_wfr, i_gfr, i_alq, :]
                
                # Строка с параметрами
                row = {
                    'THP_бар': thp_bar[i_thp],
                    'WGR_м3/м3': wgr_values[i_wfr],
                }
                
                # Добавляем BHP для каждого дебита
                for i_flo in range(NFLO):
                    row[f'Q={gas_rates[i_flo]:.0f}'] = round(bhp_values[i_flo], 4)
                
                rows_data.append(row)

# Создаём DataFrame
df_vfp = pd.DataFrame(rows_data)

# Просто выводим в консоль
print("\n" + "=" * 100)
print("VFP ТАБЛИЦА (BHP в барах)")
print("=" * 100)
print(df_vfp.to_string())
print("=" * 100)

# Если хочешь посмотреть в VSCode как таблицу
df_vfp


# ============================================================
# ЗАПИСЬ VFP ТАБЛИЦЫ В ФАЙЛ
# ============================================================

output_lines = []

# Строка 1: ключевое слово
output_lines.append("VFPPROD")

# Строка 2: основные данные таблицы
# номер_таблицы глубина_забоя тип_FLO тип_WFR тип_GFR тип_THP тип_ALQ единицы_измерения тип_данных
output_lines.append(
    f" {table_number} {max_TVD:.0f} GAS WGR OGR THP GRAT METRIC BHP /"
)

# Строка 3: значения FLO (дебиты газа)
flo_line = " " + " ".join([f"{q:.0f}" for q in gas_rates]) + " /"
output_lines.append(flo_line)

# Строка 4: значения THP (устьевые давления)
thp_line = " " + " ".join([f"{p:.0f}" for p in thp_bar]) + " /"
output_lines.append(thp_line)

# Строка 5: значения WFR (WGR)
wfr_line = " " + " ".join([f"{w:.6f}" for w in wgr_values]) + " /"
output_lines.append(wfr_line)

# Строка 6: значения GFR (OGR)
gfr_line = " " + " ".join([f"{g:.1f}" for g in ogr_values]) + " /"
output_lines.append(gfr_line)

# Строка 7: значения ALQ (GRAT)
alq_line = " " + " ".join([f"{a:.1f}" for a in grat_values]) + " /"
output_lines.append(alq_line)

# Строки с данными BHP
# Формат: NT NW NG NA BHP_1 BHP_2 ... BHP_NFLO /
print("\nЗапись результатов в файл...")

for i_thp in range(NTHP):
    for i_wfr in range(NWFR):
        for i_gfr in range(NGFR):
            for i_alq in range(NALQ):
                # Номера (индексация с 1)
                nt = i_thp + 1
                nw = i_wfr + 1
                ng = i_gfr + 1
                na = i_alq + 1
                
                # Значения BHP для всех дебитов газа
                bhp_values = bhp_results[i_thp, i_wfr, i_gfr, i_alq, :]
                
                # Форматируем строку
                bhp_str = " ".join([f"{bhp:.6f}" for bhp in bhp_values])
                data_line = f" {nt} {nw} {ng} {na} {bhp_str} /"
                output_lines.append(data_line)

# Сохраняем в файл
output_file = "vfp_table.inc"
with open(output_file, 'w', encoding='utf-8') as f:
    for line in output_lines:
        f.write(line + '\n')

print("\n" + "=" * 70)
print(f"VFP таблица сохранена в файл: {output_file}")
print(f"Количество строк в файле: {len(output_lines)}")
print("=" * 70)

# Выводим первые 15 строк для проверки
print("\nПервые 15 строк файла:")
print("-" * 70)
for i, line in enumerate(output_lines[:15]):
    print(line)
if len(output_lines) > 15:
    print("...")

# Выводим пример данных
print("\n" + "=" * 70)
print("ПРИМЕР ДАННЫХ В ТАБЛИЦЕ:")
print("=" * 70)
print(f"THP_1={thp_bar[0]} бар, WGR_1={wgr_values[0]:.6f}:")
for i_flo in range(NFLO):
    bhp = bhp_results[0, 0, 0, 0, i_flo]
    print(f"  Qgas={gas_rates[i_flo]:.0f} м3/сут -> BHP={bhp:.4f} бар")

FileNotFoundError: [Errno 2] No such file or directory: 'd:/Python_2025/Gas_liquid_flow/исходные_данные/vertical_well.dev'